In [1]:
from transformers import AutoTokenizer
from google.colab import drive

drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/NLP/resources/"

tokenizer_en = AutoTokenizer.from_pretrained(base_path + "tokenizer_en")
tokenizer_fr = AutoTokenizer.from_pretrained(base_path + "tokenizer_fr")

V_encoder = tokenizer_fr.vocab_size
V_decoder = tokenizer_en.vocab_size

fr_pad_id = tokenizer_fr.pad_token_id
en_pad_id = tokenizer_en.pad_token_id
bos_id = tokenizer_en.bos_token_id
eos_id = tokenizer_en.eos_token_id

print(f"Encoder Vocab Size V(e): {V_encoder}")
print(f"Decoder Vocab Size V(d): {V_decoder}")
print(f"Padding Token ID: {fr_pad_id}")
print(f"BOS Token ID: {bos_id}")
print(f"EOS Token ID: {eos_id}")

Mounted at /content/drive
Encoder Vocab Size V(e): 3200
Decoder Vocab Size V(d): 3200
Padding Token ID: 3
BOS Token ID: 1
EOS Token ID: 2


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk

train_hf = load_from_disk(base_path + "parallel_en_fr_corpus/train")
valid_hf = load_from_disk(base_path + "parallel_en_fr_corpus/validation")
test_hf = load_from_disk(base_path + "parallel_en_fr_corpus/test")

class NMTDataset(Dataset):
    def __init__(self, dataset_hf, tokenizer_fr, tokenizer_en, max_len=32):
        self.data = dataset_hf
        self.tok_fr = tokenizer_fr
        self.tok_en = tokenizer_en
        self.max_len = max_len

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Extract the text for the current row
        fr_text = self.data[idx]['text_fr'] 
        en_text = self.data[idx]['text_en']

        # Tokenize French (Encoder Input)
        enc_encoding = self.tok_fr(
            fr_text, 
            max_length=self.max_len, 
            padding='max_length', 
            truncation=True, 
            return_tensors="pt"
        )

        # Tokenize English (Decoder Input)
        dec_encoding = self.tok_en(
            en_text, 
            max_length=self.max_len, 
            padding='max_length', 
            truncation=True, 
            return_tensors="pt"
        )
        
        # Remove the batch dimension added by PyTorch tensors and returns IDs
        return {
            "encoder_input": enc_encoding["input_ids"].squeeze(0),
            "decoder_input": dec_encoding["input_ids"].squeeze(0)
        }
    
# Create PyTorch datasets
train_dataset = NMTDataset(train_hf, tokenizer_fr, tokenizer_en, max_len=32)
valid_dataset = NMTDataset(valid_hf, tokenizer_fr, tokenizer_en, max_len=32)
test_dataset = NMTDataset(test_hf, tokenizer_fr, tokenizer_en, max_len=32)

# Initialize DataLoaders with a batch size of 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [3]:
class TransformerEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=32):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)

    def forward(self, input_ids):
        # input_ids: (batch_size, seq_length)
        seq_length = input_ids.size(1)

        # Create position IDs for each token in the sequence
        position_ids = torch.arange(seq_length, device=input_ids.device).unsqueeze(0)
        
        token_embeds = self.token_embedding(input_ids)      # Shape: [batch_size, seq_len, d_model]
        position_embeds = self.position_embedding(position_ids)     # Shape: [1, seq_len, d_model]
        
        return token_embeds + position_embeds

In [4]:
import math
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=32, H=4):
        super().__init__()
        
        self.d_model = d_model
        self.H = H
        self.d_H = d_model // H
        
        # Linear layers for Q, K, V
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
        # Final linear layer
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        seq_length_q = query.size(1)
        seq_length_k = key.size(1)

        # Project inputs to Q, K, V
        Q = self.W_q(query)  # Shape: [batch_size, seq_length_q, d_model]
        K = self.W_k(key)    # Shape: [batch_size, seq_length_k, d_model]
        V = self.W_v(value)  # Shape: [batch_size, seq_length_k, d_model]

        # Split into heads. Shape: [batch_size, H, seq_length, d_H]
        Q = Q.view(batch_size, seq_length_q, self.H, self.d_H).transpose(1, 2)
        K = K.view(batch_size, seq_length_k, self.H, self.d_H).transpose(1, 2)
        V = V.view(batch_size, seq_length_k, self.H, self.d_H).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_H)     # Shape: [batch_size, H, seq_length_q, seq_length_k]

        # Apply masking to avoid attention to padding / future tokens
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = torch.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length_q, self.d_model)

        return self.W_o(attn_output)


In [5]:
class FFNN(nn.Module):
    def __init__(self, d_model=32, d_I=128, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_I)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(d_I, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.linear2(self.relu(self.linear1(x))))

In [6]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model=32, H=4, d_I=128, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, H)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffnn = FFNN(d_model, d_I, dropout)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        attn_output = self.attention(x, x, x, mask)
        x = self.norm1(x + attn_output)  # Add & Norm

        ffnn_output = self.ffnn(x)
        x = self.norm2(x + ffnn_output)   # Add & Norm

        return x

In [7]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model=32, H=4, d_I=128, num_layers=3, max_len=32, dropout=0.1):
        super().__init__()
        # Emedding layer (Token + Position)
        self.embeddings = TransformerEmbeddings(vocab_size, d_model, max_len)

        # Stack 3 Encoder Layers
        self.layers = nn.ModuleList([EncoderLayer(d_model, H, d_I, dropout) for _ in range(num_layers)])

    def forward(self, input_ids, mask=None):
        x = self.embeddings(input_ids)
        for layer in self.layers:
            x = layer(x, mask)
        return x    # Shape: [batch_size, seq_length, d_model]

In [8]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model=32, H=4, d_I=128, dropout=0.1):
        super().__init__()
        # Masked Self Attention
        self.self_attention = MultiHeadAttention(d_model, H)
        self.norm1 = nn.LayerNorm(d_model)

        # Encoder-Decoder Attention
        self.cross_attention = MultiHeadAttention(d_model, H)
        self.norm2 = nn.LayerNorm(d_model)

        self.ffnn = FFNN(d_model, d_I, dropout)
        self.norm3 = nn.LayerNorm(d_model)
    
    def forward(self, x, enc_output, self_mask=None, cross_mask=None):
        # Masked Self attention
        self_attn_output = self.self_attention(x, x, x, self_mask)
        x = self.norm1(x + self_attn_output)

        # Cross attention
        cross_attn_output = self.cross_attention(x, enc_output, enc_output, cross_mask)
        x = self.norm2(x + cross_attn_output)

        ffnn_output = self.ffnn(x)
        x = self.norm3(x + ffnn_output)

        return x

In [9]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=32, H=4, d_I=128, num_layers=3, max_len=32, dropout=0.1):
        super().__init__()
        self.embeddings = TransformerEmbeddings(vocab_size, d_model, max_len)
        self.layers = nn.ModuleList([DecoderLayer(d_model, H, d_I, dropout) for _ in range(num_layers)])
    
    def forward(self, input_ids, enc_output, self_mask=None, cross_mask=None):
        x = self.embeddings(input_ids)
        for layer in self.layers:
            x = layer(x, enc_output, self_mask, cross_mask)
        return x    # Shape: [batch_size, seq_length, d_model]

In [10]:
class Transformer(nn.Module):
    def __init__(self, enc_vocab_size, dec_vocab_size, enc_pad_idx, dec_pad_idx, d_model=32, H=4, d_I=128, num_layers=3, max_len=32, dropout=0.1):
        super().__init__()
        self.enc_pad_idx = enc_pad_idx  # French pad Token ID
        self.dec_pad_idx = dec_pad_idx  # English pad Token ID

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.encoder = Encoder(enc_vocab_size, d_model, H, d_I, num_layers, max_len, dropout)
        self.decoder = Decoder(dec_vocab_size, d_model, H, d_I, num_layers, max_len, dropout)
    
    def make_pad_mask(self, x, pad_idx):
        # x: [batch_size, seq_length]
        return (x != pad_idx).unsqueeze(1).unsqueeze(2)  # Shape: [batch_size, 1, 1, seq_length]
    
    def make_causal_mask(self, seq_len):
        # Create lower Triangular mask
        # Shape: [1, 1, seq_len, seq_len]
        mask = torch.tril(torch.ones((seq_len, seq_len), device=self.device, dtype=torch.bool)).view(1, 1, seq_len, seq_len)
        return mask
    
    def forward(self, enc_input_ids, dec_input_ids):
        # enc_input_ids: French tokens [Batch, seq_len]
        # dec_input_ids: English tokens [Batch, seq_len]

        # Encoder pad mask for French padding
        enc_pad_mask = self.make_pad_mask(enc_input_ids, self.enc_pad_idx)
        cross_mask = enc_pad_mask

        # Decoder self-attention mask combines English padding AND the causal mask
        dec_pad_mask = self.make_pad_mask(dec_input_ids, self.dec_pad_idx)
        causal_mask = self.make_causal_mask(dec_input_ids.size(1))
        dec_self_mask = dec_pad_mask & causal_mask 

        enc_output = self.encoder(enc_input_ids, enc_pad_mask)
        dec_output = self.decoder(dec_input_ids, enc_output, dec_self_mask, cross_mask)

        # Weight tying
        dec_embeddings_weights = self.decoder.embeddings.token_embedding.weight
        logits = torch.matmul(dec_output, dec_embeddings_weights.T)

        return logits    # Shape: [batch_size, seq_length, dec_vocab_size]

In [11]:
import torch.optim as optim
import os

def train_transformer(model, train_loader, val_loader, optimizer, criterion, device, checkpoint_dir, max_epochs=10):
    os.makedirs(checkpoint_dir, exist_ok=True)
    print(f"Starting training for {max_epochs} epochs on {device}\n")

    for epoch in range(max_epochs):
        # Training Phase
        model.train()
        train_epoch_loss = 0

        for batch_idx, batch in enumerate(train_loader):
            src = batch["encoder_input"].to(device)
            trg = batch["decoder_input"].to(device)

            # Input to decoder: All tokens except the last one
            trg_input = trg[:, :-1]

            # Expected output: All tokens except the first one
            trg_output = trg[:, 1:]

            optimizer.zero_grad()
            logits = model(src, trg_input)

            # Reshape for loss
            # Logits shape: [batch_size, seq_length, vocab_size]
            # Target shape: [batch_size * seq_length]
            vocab_size = logits.shape[-1]
            logits_flat = logits.contiguous().view(-1, vocab_size)
            trg_output_flat = trg_output.contiguous().view(-1)

            loss = criterion(logits_flat, trg_output_flat)
            loss.backward()

            # Clip gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            train_epoch_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch+1}/{max_epochs} | Batch {batch_idx}/{len(train_loader)} | Train Loss: {loss.item():.4f}")

        # Calculate average loss for the epoch
        avg_train_loss = train_epoch_loss / len(train_loader)

        # Validation Phase
        model.eval()
        valid_epoch_loss = 0
        
        # Disable gradient calculation for validation
        with torch.no_grad():
            for batch in valid_loader:
                src = batch["encoder_input"].to(device)
                trg = batch["decoder_input"].to(device)
                
                trg_input = trg[:, :-1]
                trg_target = trg[:, 1:]
                
                logits = model(src, trg_input)
                
                vocab_size = logits.shape[-1]
                logits_flat = logits.contiguous().view(-1, vocab_size)
                trg_target_flat = trg_target.contiguous().view(-1)
                
                val_loss = criterion(logits_flat, trg_target_flat)
                valid_epoch_loss += val_loss.item()
                
        avg_valid_loss = valid_epoch_loss / len(valid_loader)

        print(f"=== Epoch {epoch+1} Completed ===")
        print(f"Avg Train Loss: {avg_train_loss:.4f} | Avg Valid Loss: {avg_valid_loss:.4f}")
        
        checkpoint_path = os.path.join(checkpoint_dir, f"transformer_epoch_{epoch+1}.pt")
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'valid_loss': avg_valid_loss
        }
        torch.save(checkpoint, checkpoint_path)
        print(f"Checkpoint saved to {checkpoint_path}\n")

    print("Training Complete!")
    return model

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate the Model
model = Transformer(
    enc_vocab_size=V_encoder, 
    dec_vocab_size=V_decoder, 
    enc_pad_idx=fr_pad_id, 
    dec_pad_idx=en_pad_id
).to(device)

# Setup the Loss Function
criterion = nn.CrossEntropyLoss(ignore_index=en_pad_id)

#  Setup the Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Directory for checkpoints
checkpoint_dir = "/content/drive/MyDrive/NLP/checkpoints"
trained_model = train_transformer(
    model=model, 
    train_loader=train_loader, 
    val_loader=valid_loader, 
    optimizer=optimizer, 
    criterion=criterion, 
    device=device, 
    checkpoint_dir=checkpoint_dir
)

Starting training for 10 epochs on cuda

Epoch 1/10 | Batch 0/272 | Train Loss: 22.0182
Epoch 1/10 | Batch 100/272 | Train Loss: 5.9504
Epoch 1/10 | Batch 200/272 | Train Loss: 3.9402
=== Epoch 1 Completed ===
Avg Train Loss: 6.1970 | Avg Valid Loss: 3.5900
Checkpoint saved to /content/drive/MyDrive/NLP/checkpoints/transformer_epoch_1.pt

Epoch 2/10 | Batch 0/272 | Train Loss: 3.7015
Epoch 2/10 | Batch 100/272 | Train Loss: 3.1295
Epoch 2/10 | Batch 200/272 | Train Loss: 3.1209
=== Epoch 2 Completed ===
Avg Train Loss: 3.3815 | Avg Valid Loss: 2.9883
Checkpoint saved to /content/drive/MyDrive/NLP/checkpoints/transformer_epoch_2.pt

Epoch 3/10 | Batch 0/272 | Train Loss: 3.4481
Epoch 3/10 | Batch 100/272 | Train Loss: 3.0543
Epoch 3/10 | Batch 200/272 | Train Loss: 3.1531
=== Epoch 3 Completed ===
Avg Train Loss: 2.9600 | Avg Valid Loss: 2.7905
Checkpoint saved to /content/drive/MyDrive/NLP/checkpoints/transformer_epoch_3.pt

Epoch 4/10 | Batch 0/272 | Train Loss: 2.7746
Epoch 4/10 | Ba

In [13]:
import torch.nn.functional as F

def beam_search(model, src_tensor, src_pad_idx, bos_id, eos_id, device, max_len=32, beam_width=3):
    # src_tensor: [1, seq_len] (single sentence)
    model.eval()
    with torch.no_grad():
        # Encode the source sentence ONCE
        enc_pad_mask = model.make_pad_mask(src_tensor, src_pad_idx)
        enc_output = model.encoder(src_tensor, enc_pad_mask)

        beams = [([bos_id], 0.0)]  # (current_sequence, cumulative_log_prob)
        completed_beams = []

        for step in range(max_len):
            new_beams = []
            for seq, cum_log_prob in beams:
                if seq[-1] == eos_id:
                    completed_beams.append((seq, cum_log_prob))
                    continue
                
                # Prepare the current sequence for the decoder
                trg_tensor = torch.tensor([seq], dtype=torch.long, device=device)

                # Generate decoder masks
                trg_pad_mask = model.make_pad_mask(trg_tensor, model.dec_pad_idx)
                causal_mask = model.make_causal_mask(trg_tensor.size(1))
                self_mask = trg_pad_mask & causal_mask

                # Decoder Forward Pass
                dec_output = model.decoder(trg_tensor, enc_output, self_mask=self_mask, cross_mask=enc_pad_mask)
                dec_embeddings_weights = model.decoder.embeddings.token_embedding.weight
                logits = torch.matmul(dec_output, dec_embeddings_weights.T)

                next_token_logits = logits[0, -1, :]

                next_token_probs = F.log_softmax(next_token_logits, dim=-1)
                topk_probs, topk_ids = torch.topk(next_token_probs, beam_width)
                
                # Expand current beam into K new beams
                for i in range(beam_width):
                    new_seq = seq + [topk_ids[i].item()]
                    new_score = cum_log_prob + topk_probs[i].item()
                    new_beams.append((new_seq, new_score))

            if not new_beams:
                break
            # Keep only the top K beams
            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_width]
        
        completed_beams.extend(beams)
        # Normalize the scores by sequence length so longer sentences aren't unfairly penalized
        completed_beams = sorted(completed_beams, key=lambda x: x[1] / len(x[0]), reverse=True)

        best_sequence = completed_beams[0][0]
        return best_sequence

In [25]:
def generate_examples(model, data_loader, tokenizer_fr, tokenizer_en, device, num_examples=2):
    model.eval()
    
    # Get a single batch
    batch = next(iter(data_loader))
    src_batch = batch["encoder_input"].to(device)
    trg_batch = batch["decoder_input"].to(device)
    
    print("=== Translation Examples ===")
    
    for i in range(num_examples):
        # Extract a single sentence from the batch [1, seq_len]
        src_tensor = src_batch[i].unsqueeze(0)
        trg_tensor = trg_batch[i]
        
        # Generate translation using Beam Search
        pred_tokens = beam_search(
            model=model, 
            src_tensor=src_tensor, 
            src_pad_idx=model.enc_pad_idx, 
            bos_id=tokenizer_en.bos_token_id, 
            eos_id=tokenizer_en.eos_token_id, 
            device=device,
            max_len=32,
            beam_width=3
        )
        
        # Decode the IDs back into text (skipping <s>, </s>, and <pad>)
        src_text = tokenizer_fr.decode(src_tensor[0].tolist(), skip_special_tokens=True)
        target_text = tokenizer_en.decode(trg_tensor.tolist(), skip_special_tokens=True)
        predicted_text = tokenizer_en.decode(pred_tokens, skip_special_tokens=True)
        
        print(f"\nExample {i+1}:")
        print(f"French (Input)  : {src_text}")
        print(f"English (Target): {target_text}")
        print(f"Model Predicted : {predicted_text}")

# Run it on your training data
generate_examples(model, train_loader, tokenizer_fr, tokenizer_en, device, num_examples=2)

=== Translation Examples ===

Example 1:
French (Input)  : ▁je ▁me ▁rejouis ▁de ▁vous ▁voir ▁heureux ▁.
English (Target): ▁i ▁m ▁glad ▁to ▁see ▁you ▁re ▁happy ▁.
Model Predicted : ▁i ▁m ▁glad ▁to ▁see ▁you ▁are ▁.

Example 2:
French (Input)  : ▁je ▁suis ▁un ▁peu ▁bour @@ ▁ree ▁.
English (Target): ▁i ▁m ▁a ▁little ▁drunk ▁.
Model Predicted : ▁i ▁m ▁a ▁little ▁me ▁.


In [ ]:
import nltk
from nltk.translate.bleu_score import corpus_bleu
import tqdm

def calculate_bleu(model, data_loader, tokenizer_en, device, max_samples=200):
    """
    Calculates the BLEU score over the dataset using Beam Search.
    """
    model.eval()
    references = []
    hypotheses = []
    
    print(f"Calculating BLEU Score over {max_samples} samples...")
    
    count = 0
    for batch in data_loader:
        if count >= max_samples:
            break
            
        src_batch = batch["encoder_input"].to(device)
        trg_batch = batch["decoder_input"].to(device)
        
        # Process each sequence in the batch individually
        for i in range(src_batch.size(0)):
            if count >= max_samples:
                break
                
            src_tensor = src_batch[i].unsqueeze(0)
            trg_tensor = trg_batch[i]
            
            # Predict
            pred_tokens = beam_search(
                model=model, src_tensor=src_tensor, src_pad_idx=model.enc_pad_idx, 
                bos_id=tokenizer_en.bos_token_id, eos_id=tokenizer_en.eos_token_id, 
                device=device
            )
            
            # Decode to strings
            target_text = tokenizer_en.decode(trg_tensor.tolist(), skip_special_tokens=True)
            predicted_text = tokenizer_en.decode(pred_tokens, skip_special_tokens=True)
            
            references.append([target_text.split()])
            hypotheses.append(predicted_text.split())
            
            count += 1
            
    # Calculate corpus BLEU
    bleu_score = corpus_bleu(references, hypotheses)
    print(f"\nCorpus BLEU Score: {bleu_score * 100:.2f}")
    return bleu_score

test_bleu = calculate_bleu(model, test_loader, tokenizer_en, device, max_samples=500)

Calculating BLEU Score over 500 samples...

Corpus BLEU Score: 22.41
